In [ ]:
from pathlib import Path
import sys


SCENARIO_DIR = Path.cwd()
REPO_ROOT = SCENARIO_DIR.parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modules.security.models import SecurityModelType
from pipelines.routing import CGRYenRouting
from pipelines.simulation import SimulationPipeline


SERIES_COLORS = {
    SecurityModelType.HOP_BY_HOP: "#0f4c5c",
    SecurityModelType.END_TO_END: "#e36414",
    SecurityModelType.EDGE_BY_EDGE: "#6a994e",
    SecurityModelType.EDGE_TO_EDGE: "#8d0801",
}

max_routes = 1
pipeline = SimulationPipeline()

result = pipeline.run(
    cp_path=str(SCENARIO_DIR / "contact_plan.json"),
    topology_path=str(SCENARIO_DIR / "topology.json"),
    security_models=tuple(SecurityModelType),
    curr_time=0,
    routing_algorithm=CGRYenRouting(max_routes=max_routes),
)

In [ ]:
def print_pair_data(pair):
    print(','.join([f"{node:3d}" for node in result.annotation.annotated_routes_by_pair[pair][0].node_path]))
    print(','.join([f"{net:3d}" for net in result.annotation.annotated_routes_by_pair[pair][0].network_path]))
    print(f"Boundries: {result.annotation.annotated_routes_by_pair[pair][0].boundary_crossings}")
    print(f"HBH: {" ".join([f"{req}" for req in result.security.plans_by_model[SecurityModelType.HOP_BY_HOP][pair][0].key_requirements])}")
    print(f"E2E: {" ".join([f"{req}" for req in result.security.plans_by_model[SecurityModelType.END_TO_END][pair][0].key_requirements])}")
    print(f"EBE: {" ".join([f"{req}" for req in result.security.plans_by_model[SecurityModelType.EDGE_BY_EDGE][pair][0].key_requirements])}")
    print(f"ETE: {" ".join([f"{req}" for req in result.security.plans_by_model[SecurityModelType.EDGE_TO_EDGE][pair][0].key_requirements])}")

In [ ]:
print_pair_data((1,12))

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

MODEL_PALETTE = {
    "HOP_BY_HOP": "#4c78a8",
    "END_TO_END": "#f58518",
    "EDGE_BY_EDGE": "#54a24b",
    "EDGE_TO_EDGE": "#e45756",
    "hbh": "#4c78a8",
    "e2e": "#f58518",
    "ebe": "#54a24b",
    "ete": "#e45756",
    "hop_by_hop": "#4c78a8",
    "end_to_end": "#f58518",
    "edge_by_edge": "#54a24b",
    "edge_to_edge": "#e45756",
}


def palette_for(values):
    unique_values = list(dict.fromkeys(values))
    return {value: MODEL_PALETTE.get(value, "#4c78a8") for value in unique_values}

rows = []

for model, plans_by_pair in result.security.plans_by_model.items():
    for pair, plans in plans_by_pair.items():
        if pair==(1,12):
            for plan in plans:
                unique_scopes = {req.scope for req in plan.key_requirements}
                rows.append(
                    {
                        "model": model.name.lower(),
                        "pair": f"{pair[0]}->{pair[1]}",
                        "route": f"{pair[0]}->{pair[1]}:{plan.route_id}",
                        "required_keys": len(unique_scopes),
                    }
                )

df_keys = pd.DataFrame(rows)
plt.figure(figsize=(8, 5))
sns.boxenplot(
    data=df_keys,
    x="model",
    y="required_keys",
    hue="model",
    dodge=False,
    palette=palette_for(df_keys["model"]),
    legend=False,
)
plt.title("Distribution of required keys by model")
plt.xlabel("Model")
plt.ylabel("Required keys per route")
plt.tight_layout()
plt.show()
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

MODEL_PALETTE = {
    "HOP_BY_HOP": "#4c78a8",
    "END_TO_END": "#f58518",
    "EDGE_BY_EDGE": "#54a24b",
    "EDGE_TO_EDGE": "#e45756",
    "hbh": "#4c78a8",
    "e2e": "#f58518",
    "ebe": "#54a24b",
    "ete": "#e45756",
    "hop_by_hop": "#4c78a8",
    "end_to_end": "#f58518",
    "edge_by_edge": "#54a24b",
    "edge_to_edge": "#e45756",
}


def palette_for(values):
    unique_values = list(dict.fromkeys(values))
    return {value: MODEL_PALETTE.get(value, "#4c78a8") for value in unique_values}

from modules.security.models import SecurityModelType
from pipelines.route_activation import RouteActivationPlanner

planner = RouteActivationPlanner()

rows = []
for label, sim_result in [("Directed", result)]:
    for model in SecurityModelType:
        rows.append(
            {
                "mode": label,
                "model": model.name,
                "unique_keys": len(planner.build_for_model(sim_result.security, model=model).key_scopes),
            }
        )

df = pd.DataFrame(rows)

pivot = df.pivot(index="model", columns="mode", values="unique_keys").reset_index()
pivot = pivot.sort_values("Directed", ascending=False)

plt.figure(figsize=(8, 5))
ax = sns.barplot(
    data=pivot,
    x="model",
    y="Directed",
    hue="model",
    dodge=False,
    palette=palette_for(pivot["model"]),
    legend=False,
)
ax.set_ylabel("Number of distinct keys")
ax.set_xlabel("Model")

for i, value in enumerate(pivot["Directed"]):
    ax.text(i, value + 0.3, str(value), ha="center", va="bottom")

plt.tight_layout()
plt.show()



## Maximize routes subject to a key budget

:::: {layout="[0.5, 0.5]"}

:::{#firstcol}
\begin{aligned}
P &: \text{set of reachable source-destination pairs in the network} \\
R_p &: \text{set of candidate routes for pair } p \in P \\
R &= \bigcup_{p \in P} R_p \\
K &: \text{set of available candidate keys} \\
S_r &\subseteq K : \text{set of keys required by route } r \in R \\
B &: \text{maximum budget of active keys} \\
\end{aligned}
:::

:::{#secondcol}
\begin{aligned}
\max \quad & \sum_{r \in R} x_r \\
\text{s.t.} \quad
& \sum_{k \in K} y_k \le B \\
& \sum_{k \in S_r} y_k \ge |S_r|\,x_r \qquad \forall r \in R \\
& x_r \in \{0,1\} \qquad \forall r \in R \\
& y_k \in \{0,1\} \qquad \forall k \in K
\end{aligned}
:::


::::


In [ ]:
import pandas as pd

df_m1 = pd.DataFrame()
traces_m1 = {}
for sec_model in SERIES_COLORS.keys():
    from models.model1_max_routes import solve_for_security_model

    sec_traces, sec_df = solve_for_security_model(result, sec_model)
    traces_m1[sec_model.name] = dict(zip(sec_df["max_keys"], sec_traces))
    df_m1 = pd.concat([df_m1, sec_df])


In [ ]:
#| echo: false
plt.figure(figsize=(9, 6))
ax = sns.lineplot(
    data=df_m1.sort_values("selected_keys"),
    x="selected_keys",
    y="selected_routes",
    hue="model",
    linewidth=2,
    palette=palette_for(df_m1["model"]),
)
ax.set_xlabel("Active keys")
ax.set_ylabel("Enabled routes")
ax.legend(title="Model")
plt.show()


## Minimize keys for a pair-connectivity target


:::: {layout="[0.5, 0.5]"}

:::{#firstcol}

\begin{aligned}
P &: \text{set of reachable source-destination pairs in the network} \\
R_p &: \text{set of candidate routes for pair } p \in P \\
R &= \bigcup_{p \in P} R_p \\
K &: \text{set of available candidate keys} \\
S_r &\subseteq K : \text{set of keys required by route } r \in R \\
\alpha &: \text{target connectivity fraction or percentage}
\end{aligned}
$$
x_r =
\begin{cases}
1 & \text{if route } r \text{ is selected} \\
0 & \text{otherwise}
\end{cases}
\qquad
y_k =
\begin{cases}
1 & \text{if key } k \text{ is available} \\
0 & \text{otherwise}
\end{cases}
$$
$$
z_p =
\begin{cases}
1 & \text{if pair } p \text{ has at least 1 route } \\
0 & \text{otherwise}
\end{cases}
$$

:::

:::{#secondcol}


\begin{aligned}
\min \quad & \sum_{k \in K} y_k \\
\text{s.t.} \quad
& \sum_{k \in S_r} y_k \ge |S_r|\,x_r \qquad \forall r \in R \\
& x_r \le z_p \qquad \forall p \in P,\ \forall r \in R_p \\
& z_p \le \sum_{r \in R_p} x_r \qquad \forall p \in P \\
& \sum_{p \in P} z_p \ge \left\lceil \alpha |P| \right\rceil \\
& x_r \in \{0,1\} \qquad \forall r \in R \\
& y_k \in \{0,1\} \qquad \forall k \in K \\
& z_p \in \{0,1\} \qquad \forall p \in P
\end{aligned}

:::


::::


In [ ]:
import pandas as pd


df_m2 = pd.DataFrame()
traces_m2 = {}

for sec_model in SERIES_COLORS.keys():
    from models.model2_min_keys import solve_for_security_model

    sec_traces, sec_df = solve_for_security_model(result, sec_model)
    traces_m2[sec_model.name] = dict(zip(sec_df["target_connectivity_pct"], sec_traces))
    df_m2 = pd.concat([df_m2, sec_df], ignore_index=True)


In [ ]:
#| echo: false
plt.figure(figsize=(9, 6))
ax = sns.lineplot(
    data=df_m2.sort_values("target_connectivity_pct"),
    x="target_connectivity_pct",
    y="selected_keys",
    hue="model",
    linewidth=2,
    palette=palette_for(df_m2["model"]),
)
ax.set_xlabel("Target connectivity (%)")
ax.set_ylabel("Minimum number of active keys")
ax.set_title("Minimum keys vs target connectivity by model")
ax.legend(title="Model")
plt.show()


## Minimize keys while guaranteeing at least 1 path per pair


:::: {layout="[0.5, 0.5]"}

:::{#firstcol}

\begin{aligned}
P &: \text{set of reachable source-destination pairs in the network} \\
R_p &: \text{set of candidate routes for pair } p \in P \\
R &= \bigcup_{p \in P} R_p \\
K &: \text{set of available candidate keys} \\
S_r &\subseteq K : \text{set of keys required by route } r \in R \\
\end{aligned}
$$
x_r =
\begin{cases}
1 & \text{if route } r \text{ is selected} \\
0 & \text{otherwise}
\end{cases}
\qquad
y_k =
\begin{cases}
1 & \text{if key } k \text{ is available} \\
0 & \text{otherwise}
\end{cases}
$$
:::

:::{#secondcol}


\begin{aligned}
\min \quad & \sum_{k \in K} y_k \\
\text{s.t.} \quad
& \sum_{k \in S_r} y_k \ge |S_r|\,x_r \qquad \forall r \in R \\
& 1 \le \sum_{r \in R_p} x_r \qquad \forall p \in P \\
& x_r \in \{0,1\} \qquad \forall r \in R \\
& y_k \in \{0,1\} \qquad \forall k \in K \\
\end{aligned}

:::


::::


In [ ]:
import pandas as pd


df_m3 = pd.DataFrame()
traces_m3 = {}

for sec_model in SERIES_COLORS.keys():
    from models.model3_min_keys_full_connectivity import solve_for_security_model

    sec_traces, sec_df = solve_for_security_model(result, sec_model)
    traces_m3[sec_model.name] = sec_traces[0] if sec_traces else None
    df_m3 = pd.concat([df_m3, sec_df], ignore_index=True)


In [ ]:
#| label: fig-gapminder
#| fig-cap: "Model 3 results"
#| fig-subcap:
#|   - "Minimum keys to guarantee at least 1 path per pair"
#|   - "Routes selected in the optimal solution"
#| layout-ncol: 2
#| column: page
#| echo: false
#| out-width: 100%

plot_df = df_m3.sort_values("selected_keys").copy()

plt.figure(figsize=(9, 5))
ax = sns.barplot(
    data=plot_df,
    x="model",
    y="selected_keys",
    hue="model",
    dodge=False,
    palette=palette_for(plot_df["model"]),
    legend=False,
)
ax.set_xlabel("Security model")
ax.set_ylabel("Minimum number of keys")

for i, value in enumerate(plot_df["selected_keys"]):
    ax.text(i, value + 0.05, str(value), ha="center")

plt.tight_layout()
plt.show()

plot_df = df_m3.sort_values("selected_routes").copy()

plt.figure(figsize=(9, 5))
ax = sns.barplot(
    data=plot_df,
    x="model",
    y="selected_routes",
    hue="model",
    dodge=False,
    palette=palette_for(plot_df["model"]),
    legend=False,
)
ax.set_xlabel("Security model")
ax.set_ylabel("Number of selected routes")

for i, value in enumerate(plot_df["selected_routes"]):
    ax.text(i, value + 0.05, str(value), ha="center")

plt.tight_layout()
plt.show()
